In [1]:
import pandas as pd
import numpy as np
import tqdm
from typing import *

import numpy as np
import pandas as pd
from IPython.display import *
from call_bedrock_fast_claude_v3_py import *
#import sagemaker
import boto3
#from sagemaker import get_execution_role

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold
#import sagemaker
import boto3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler,MinMaxScaler
import matplotlib.pyplot as plt
#import torch
from transformers import BertTokenizer, BertModel
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import faiss
#from titan_embedding_first import *
import awswrangler as wr
#from titan_parallel import *
from haiku_parallel import *

In [2]:
base=pd.read_csv('s3://biswa--test/Unsaved/2025/09/16/44b53a39-9a46-49b0-874b-8dffd2922e4f.csv')

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,asin,brand_name,gl_product_group_desc,item_name,product_category,product_subcategory,product_type,product_description,PL
0,B0D9BR2H3C,GLADTOGIFT,gl_office_product,G2g Matte Paper Pattern Scrapbooking 12x12 pap...,22912000.0,22912001.0,SHEET_PAPER,Elevate your DIY crafts and scrapbooking proje...,NaN
1,B0CHFRPYQH,VRB Dec,gl_home,VRB Dec™ Acrylic Flameless & Smokeless Decorat...,20106500.0,20106504.0,LAMP,PERFECT FOR : Beautiful decoration night light...,Other_Hardlines
2,B0DPCNC1JG,Generic,gl_home,Thalapathy Vijay: A Legacy of Hits Across Cine...,20106500.0,20106514.0,PICTURE_FRAME,Celebrate the illustrious career of Thalapathy...,Other_Hardlines
3,B0D9YNK88J,Apka Mart The Online Shop,gl_home,Apka Mart The Online Shop Lord Ram Wall Hangin...,20106500.0,20106508.0,FIGURINE,This exquisitely crafted Ram Lalla wall hangin...,Other_Hardlines
4,B07CQ29BHX,Mack Jonney,gl_apparel,Mack JONNEY Men's Slim Fit Trackpants (DD7NAVY...,19301000.0,19301031.0,PANTS,Lay a hand on this slim fit camisole presented...,Softlines
...,...,...,...,...,...,...,...,...,...
5408923,B0CNPZTJBG,KRYPTIC,gl_apparel,KRYPTIC Men Black Botanical Printed Polo Colla...,19301600.0,19301605.0,SHIRT,Kryptic 100% Cotton Knitted Botanical Printed ...,Softlines
5408924,B0DVH2M5NN,GolfBasic,gl_apparel,GolfBasic 60'' Lightweight Single Canopy Auto ...,NaN,NaN,UMBRELLA,The GolfBasic EP Coated Single Canopy Auto Ope...,Softlines
5408925,B0C81NT1FF,Aimly,gl_apparel,Aimly Women's Regular Fit Sleeveless Cotton Ca...,19302400.0,19302440.0,SHIRT,We care your intimacy. Keep yourself relaxed a...,Softlines
5408926,B0C1JPQXWH,LEWEL,gl_apparel,LEWEL Men's Stylish Solid Printed Full Sleeve ...,19301600.0,19301621.0,SHIRT,Step up your Style Quotient by wearing this ca...,Softlines


In [28]:
path='s3://gst2.0.0/s5_19Sep25/newop12_hsngst.csv'
cat_data=pd.read_csv(path)
print(cat_data.shape)
cat_data=base[base['asin'].isin(cat_data['asin'])]
cat_data.drop_duplicates(inplace=True)

(5608, 1)


/tmp/ipykernel_6082/1782743980.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cat_data.drop_duplicates(inplace=True)


In [29]:
display(cat_data.head(5))
cat_data.shape

,asin,brand_name,gl_product_group_desc,item_name,product_category,product_subcategory,product_type,product_description,PL
229,B07R3XWCP6,Tombow,gl_office_product,"Tombow Mono Graph Mechanical Pencil Lead, HB 0...",22908000.0,22908010.0,WRITING_INSTRUMENT,"These high-quality leads offer smooth writing,...",Hardlines
1080,B0B1QKT5F5,Scrikss,gl_office_product,Scrikss 0.7 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines
2970,B0B1QL2CMP,Scrikss,gl_office_product,Scrikss 0.5 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines
3246,B07JBD6N3W,papergrid,gl_office_product,"Papergrid Practical Notebook - Science, 28 cm ...",22903000.0,22903203.0,BLANK_BOOK,NaN,Hardlines
3387,B0C4LVV69X,PAPERDOM,gl_office_product,PAPERDOM WEEKLY WELLNESS LARGE NOTEPAD,22903000.0,22903203.0,BLANK_BOOK,""" Size: 5 x 7 inches Colour: Grey with a brown...",Hardlines


(5608, 9)

In [18]:
system='''You are a GST commodity keyword matcher. For each product, read the title inside <tit></tit> and the product description inside <pd></pd>.\
From the keyword phrase list inside <att></att>, pick the single closest phrase (or “none”).\
Then decide if the product truly falls under that phrase (subcategory decision), and return a confidence score and rationale.

Rules:
Only use the phrases exactly as written in <att>.
Respect exclusions like “other than” or “not containing”. If the product falls into an exclusion, the answer is no.
Prefer product title over description if there is a conflict.
Never hallucinate new phrases, HS codes, or ingredients.
Avoid linguistic proximity completely
Do not perform aproximate match between words. The match should always be Product defined in title to Product defined in Phrase.

Decision Process:
Identify the domain terms like toothbrush, malt, thermometer.
Collect positive evidence from <tit> and <pd>. Look for exact product type matches, synonyms, or strong signals.
Apply exclusion rules strictly. If excluded, force no with low confidence.
Choose the top-1 phrase. If ties, pick the most specific match from title.
output: yes only if product clearly belongs under the phrase and exclusions do not apply; else no.

Output format (exact):
<output></output> this is where you need to tell your decision as yes/no
<matched_phrase_id>{1-12 or "none"}</matched_phrase_id>
<matched_phrase_text>{exact phrase text or "none"}</matched_phrase_text>
<conf>{0.00–1.00}</conf>
<rat>{2–3 short lines explaining evidence, mention exclusions if checked, and why this confidence level was chosen}</rat>


Keyword phrases:
<att>
Exercise book, graph book, & laboratory note book and
notebooks
Maps and hydrographic or similar charts of all kinds, \
including atlases, wall maps, topographical plans and \
globes, printed
Pencil sharpeners
Pencils (including propelling or sliding pencils), crayons, \
pastels, drawing charcoals and tailor’s chalk

</att>
If product info is too generic or empty, return none with confidence 0.00 and rationale “insufficient information”.
Example expected behavior:
Oral-B Toothbrush → id=4, text=Tooth brushes including dental-plate brushes, decision=yes, conf=0.90, rat=title and description clearly state toothbrush.
Lipoma cream → id=none, text=none, decision=no, conf=0.10, rat=no evidence for dental, shaving, malt, confectionery, thermometer, or instruments.
Sesame chikki → id=7, text=Sugar confectionery …, decision=no, conf=0.30, rat=product is confectionery but explicitly excluded as "other than" in bracket.
Malt extract powder → id=6, text=Malt extract…, decision=yes, conf=0.80, rat=title states malt extract; no cocoa mentioned; assumption leads to strong but not max confidence.

Return only the output mentioned in output format. Do not generate anything else.'''.strip()

prompt2='''
<tit>{b}</tit>
<pt>{a}</pt>'''.strip()


In [19]:
print(system)

You are a GST commodity keyword matcher. For each product, read the title inside <tit></tit> and the product description inside <pd></pd>.From the keyword phrase list inside <att></att>, pick the single closest phrase (or “none”).Then decide if the product truly falls under that phrase (subcategory decision), and return a confidence score and rationale.

Rules:
Only use the phrases exactly as written in <att>.
Respect exclusions like “other than” or “not containing”. If the product falls into an exclusion, the answer is no.
Prefer product title over description if there is a conflict.
Never hallucinate new phrases, HS codes, or ingredients.
Avoid linguistic proximity completely
Do not perform aproximate match between words. The match should always be Product defined in title to Product defined in Phrase.

Decision Process:
Identify the domain terms like toothbrush, malt, thermometer.
Collect positive evidence from <tit> and <pd>. Look for exact product type matches, synonyms, or strong

In [30]:
#new_be11=cat_data.sample(100)
new_be11=cat_data#.iloc[0:100,:]
new_be11.reset_index(drop=True, inplace=True)
new_be11['prompt2']=new_be11.apply(lambda x:prompt2.format(b=x['item_name'],a=x['product_description']),axis=1)
responses = run_parallel_series(new_be11['prompt2'],system_message=system, max_tokens=500, temperature=0,max_workers=20)

/tmp/ipykernel_6082/737425457.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_be11['prompt2']=new_be11.apply(lambda x:prompt2.format(b=x['item_name'],a=x['product_description']),axis=1)
Processing Prompts: 100%|██████████| 5608/5608 [12:08<00:00,  7.70it/s]


In [32]:
new_be11['class']=responses
from bs4 import BeautifulSoup as soup
def ext(x,tag_name='p'):
    r=soup(str(x))
    c=r.find_all(name=tag_name)
    return str(c)
new_be11['rational']=new_be11['class'].apply(ext,tag_name='rat')
new_be11['output']=new_be11['class'].apply(ext,tag_name='output').astype('str')
new_be11['confidence']=new_be11['class'].apply(ext,tag_name='conf')
#new_be11['matched_phrase_id']=new_be11['class'].apply(ext,tag_name='matched_phrase_id')
new_be11['matched_phrase_text']=new_be11['class'].apply(ext,tag_name='matched_phrase_text').astype('str')

#res2['class1'][2]
#soup(strres2['class1'][0]).find_all(name='output')
new_be11['rational']=new_be11['rational'].astype('str').replace(['\[','\]','<rat>','</rat>'],'',regex=True)
new_be11['output']=new_be11['output'].astype('str').replace(['\[','\]','<output>','</output>'],'',regex=True)
new_be11['confidence']=new_be11['confidence'].astype('str').replace(['\[','\]','<conf>','</conf>'],'',regex=True)
#new_be11['matched_phrase_id']=new_be11['matched_phrase_id'].astype('str').replace(['\[','\]','<matched_phrase_id>','</matched_phrase_id>'],'',regex=True)
new_be11['matched_phrase_text']=new_be11['matched_phrase_text'].astype('str').replace(['\[','\]','<matched_phrase_text>','</matched_phrase_text>'],'',regex=True)
path=path.replace('s5_19Sep25','s5_19sep25_output')
new_be11.to_csv(path,index=False)
new_be11

/tmp/ipykernel_6082/2591795495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_be11['class']=responses
/tmp/ipykernel_6082/2591795495.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_be11['rational']=new_be11['class'].apply(ext,tag_name='rat')
/tmp/ipykernel_6082/2591795495.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pan

,asin,brand_name,gl_product_group_desc,item_name,product_category,product_subcategory,product_type,product_description,PL,prompt2,class,rational,output,confidence,matched_phrase_text
0,B07R3XWCP6,Tombow,gl_office_product,"Tombow Mono Graph Mechanical Pencil Lead, HB 0...",22908000.0,22908010.0,WRITING_INSTRUMENT,"These high-quality leads offer smooth writing,...",Hardlines,"<tit>Tombow Mono Graph Mechanical Pencil Lead,...",<output>yes</output>\n<matched_phrase_id>4</ma...,"Product is a mechanical pencil lead, which dir...",yes,0.95,Pencils (including propelling or sliding penci...
1,B0B1QKT5F5,Scrikss,gl_office_product,Scrikss 0.7 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines,<tit>Scrikss 0.7 mm Cute Aesthetic Mechanical ...,<output>yes</output>\n<matched_phrase_id>4</ma...,Product is a mechanical pencil (clutch lead pe...,yes,0.95,Pencils (including propelling or sliding penci...
2,B0B1QL2CMP,Scrikss,gl_office_product,Scrikss 0.5 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines,<tit>Scrikss 0.5 mm Cute Aesthetic Mechanical ...,<output>yes</output>\n<matched_phrase_id>4</ma...,Product is a mechanical pencil (0.5 mm) explic...,yes,0.95,Pencils (including propelling or sliding penci...
3,B07JBD6N3W,papergrid,gl_office_product,"Papergrid Practical Notebook - Science, 28 cm ...",22903000.0,22903203.0,BLANK_BOOK,NaN,Hardlines,"<tit>Papergrid Practical Notebook - Science, 2...",<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a practical notebook specifically d...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."
4,B0C4LVV69X,PAPERDOM,gl_office_product,PAPERDOM WEEKLY WELLNESS LARGE NOTEPAD,22903000.0,22903203.0,BLANK_BOOK,""" Size: 5 x 7 inches Colour: Grey with a brown...",Hardlines,<tit>PAPERDOM WEEKLY WELLNESS LARGE NOTEPAD</t...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notepad with specific design feat...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5603,B09JKNY2GL,NEORAH,gl_office_product,NEORAH — A5 Ruled Classic Notebook -110 Gsm (2...,22903000.0,22903203.0,BLANK_BOOK,A5 Classic premium notebook,Hardlines,<tit>NEORAH — A5 Ruled Classic Notebook -110 G...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a hardcover notebook with 160 pages...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."
5604,B09WRG94CJ,Neel,gl_office_product,Neel® 1pcs Fur Diary for Girls Personal A5 Siz...,22903000.0,22903203.0,BLANK_BOOK,diary fur diary for grils notebook fur cute so...,Hardlines,<tit>Neel® 1pcs Fur Diary for Girls Personal A...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is clearly a notebook/diary with ruled...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
5605,B082S57DTX,ImpressiveWorks,gl_office_product,ImpressiveWorks Magnetic Memo Notepad with Mag...,22903000.0,22903203.0,BLANK_BOOK,Magnetic Paper Notepad for all your important ...,Hardlines,<tit>ImpressiveWorks Magnetic Memo Notepad wit...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notepad with sheets for writing n...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
5606,B09HBVRD51,Unigo,gl_office_product,Unigo Reusable Notebook with 2 Erasable Marker...,22903000.0,22903203.0,BLANK_BOOK,Upgrade your writing experience with this reus...,Hardlines,<tit>Unigo Reusable Notebook with 2 Erasable M...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notebook with pages designed for ...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."


In [33]:
new_be11[new_be11['output']=='yes']

,asin,brand_name,gl_product_group_desc,item_name,product_category,product_subcategory,product_type,product_description,PL,prompt2,class,rational,output,confidence,matched_phrase_text
0,B07R3XWCP6,Tombow,gl_office_product,"Tombow Mono Graph Mechanical Pencil Lead, HB 0...",22908000.0,22908010.0,WRITING_INSTRUMENT,"These high-quality leads offer smooth writing,...",Hardlines,"<tit>Tombow Mono Graph Mechanical Pencil Lead,...",<output>yes</output>\n<matched_phrase_id>4</ma...,"Product is a mechanical pencil lead, which dir...",yes,0.95,Pencils (including propelling or sliding penci...
1,B0B1QKT5F5,Scrikss,gl_office_product,Scrikss 0.7 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines,<tit>Scrikss 0.7 mm Cute Aesthetic Mechanical ...,<output>yes</output>\n<matched_phrase_id>4</ma...,Product is a mechanical pencil (clutch lead pe...,yes,0.95,Pencils (including propelling or sliding penci...
2,B0B1QL2CMP,Scrikss,gl_office_product,Scrikss 0.5 mm Cute Aesthetic Mechanical Clutc...,20108600.0,20108611.0,WRITING_INSTRUMENT,"Launched in 2016, the Hexagon-R range brings a...",Hardlines,<tit>Scrikss 0.5 mm Cute Aesthetic Mechanical ...,<output>yes</output>\n<matched_phrase_id>4</ma...,Product is a mechanical pencil (0.5 mm) explic...,yes,0.95,Pencils (including propelling or sliding penci...
3,B07JBD6N3W,papergrid,gl_office_product,"Papergrid Practical Notebook - Science, 28 cm ...",22903000.0,22903203.0,BLANK_BOOK,NaN,Hardlines,"<tit>Papergrid Practical Notebook - Science, 2...",<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a practical notebook specifically d...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."
4,B0C4LVV69X,PAPERDOM,gl_office_product,PAPERDOM WEEKLY WELLNESS LARGE NOTEPAD,22903000.0,22903203.0,BLANK_BOOK,""" Size: 5 x 7 inches Colour: Grey with a brown...",Hardlines,<tit>PAPERDOM WEEKLY WELLNESS LARGE NOTEPAD</t...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notepad with specific design feat...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5603,B09JKNY2GL,NEORAH,gl_office_product,NEORAH — A5 Ruled Classic Notebook -110 Gsm (2...,22903000.0,22903203.0,BLANK_BOOK,A5 Classic premium notebook,Hardlines,<tit>NEORAH — A5 Ruled Classic Notebook -110 G...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a hardcover notebook with 160 pages...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."
5604,B09WRG94CJ,Neel,gl_office_product,Neel® 1pcs Fur Diary for Girls Personal A5 Siz...,22903000.0,22903203.0,BLANK_BOOK,diary fur diary for grils notebook fur cute so...,Hardlines,<tit>Neel® 1pcs Fur Diary for Girls Personal A...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is clearly a notebook/diary with ruled...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
5605,B082S57DTX,ImpressiveWorks,gl_office_product,ImpressiveWorks Magnetic Memo Notepad with Mag...,22903000.0,22903203.0,BLANK_BOOK,Magnetic Paper Notepad for all your important ...,Hardlines,<tit>ImpressiveWorks Magnetic Memo Notepad wit...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notepad with sheets for writing n...,yes,0.85,"Exercise book, graph book, &amp; laboratory no..."
5606,B09HBVRD51,Unigo,gl_office_product,Unigo Reusable Notebook with 2 Erasable Marker...,22903000.0,22903203.0,BLANK_BOOK,Upgrade your writing experience with this reus...,Hardlines,<tit>Unigo Reusable Notebook with 2 Erasable M...,<output>yes</output>\n<matched_phrase_id>1</ma...,Product is a notebook with pages designed for ...,yes,0.95,"Exercise book, graph book, &amp; laboratory no..."
